# Geneset / Pathway Activity

Consolidated notebook for the 6-step pipeline that previously lived in 8 separate notebooks:

1. Load an `adata` (single-cell or metacell; male, female, or full)
2. Build a geneset of interest (custom gene list, or one/more GMT files) and report overlap with `adata.var_names`
3. Score each cell with `decoupler.mt.aucell`
4. Violin–box plot of the score per group (via `pygenelab.plot_violin_box_combo`)
5. Driver genes per group via Spearman correlation → heatmap
6. Cliff's delta table between two groups

All functions live in `pygenelab.geneset_activity`. To switch dataset or pathway: uncomment one option in the **ADATA** cell and one in the **GENESET** cell, then run the rest.

In [ ]:
import sys
from pathlib import Path

# make pygenelab importable from this folder
PYGENELAB_PARENT = Path("/ix/djishnu/Akanksha/analysis_code/snRNA_TA_muscle_analysis/TA_muscle_code/py_scripts")
if str(PYGENELAB_PARENT) not in sys.path:
    sys.path.insert(0, str(PYGENELAB_PARENT))

import pygenelab as pgl
import scanpy as sc
import decoupler as dc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Cell A — pick an `adata` input

Uncomment exactly one. Each option carries its own `group_col`, `groups`, palette, and `gene_origin` so the rest of the notebook needs no edits when you switch dataset.
CRC paths are primary; PSC equivalents are commented inline for portability.

In [ ]:
# (path, group_col, group1, group2, palette, gene_origin, prep)
#   prep: optional callable f(adata) -> None for per-dataset fixups
#         (e.g. seacell metacells store condition as 'y' = 0/1 and need a mapping)

def _seacell_y_to_condition(ad):
    ad.obs["condition"] = ad.obs["y"].apply(lambda x: "WT" if x == 0 else "KO").astype("category")

# --- Mice female metacells, FastIIB / FastIIX (WT vs KO) ---
ADATA_CFG = (
    "/ix/djishnu/Akanksha/datasets/ERCC1_KO/objects/objects/seacells_F_FastIIB_FastIIX_harmonyv2.h5ad",
    # PSC: /ocean/projects/cis240075p/asachan/datasets/TA_muscle/ERCC1_KO_mice/integrated_samples/objects/seacells_F_FastIIB_FastIIX_harmonyv2.h5ad
    "condition", "WT", "KO",
    {"WT": "#B6D7A8", "KO": "#F08080"},
    "mice",
    _seacell_y_to_condition,
)

# --- Mice male metacells, FastIIB / FastIIX (WT vs KO) ---
# ADATA_CFG = (
#     "/ix/djishnu/Akanksha/datasets/ERCC1_KO/objects/objects/seacells_M_FastIIB_FastIIX_harmonyv2.h5ad",
#     # PSC: /ocean/projects/cis240075p/asachan/datasets/TA_muscle/ERCC1_KO_mice/integrated_samples/objects/seacells_M_FastIIB_FastIIX_harmonyv2.h5ad
#     "condition", "WT", "KO",
#     {"WT": "#B6D7A8", "KO": "#F08080"},
#     "mice",
#     _seacell_y_to_condition,
# )

# --- Mice full single-cell, all cell types (WT vs KO) — has 'condition' already ---
# ADATA_CFG = (
#     "/ix/djishnu/Akanksha/datasets/ERCC1_KO/objects/objects/adata_harmony_celloracle_compatible_v2.h5ad",
#     # PSC: /ocean/projects/cis240075p/asachan/datasets/TA_muscle/ERCC1_KO_mice/integrated_samples/objects/adata_harmony_celloracle_compatible_v2.h5ad
#     "condition", "WT", "KO",
#     {"WT": "#B6D7A8", "KO": "#F08080"},
#     "mice",
#     None,
# )

# --- Human female single-cell myofibers (OM6 vs OM9) — PSC-only path so far ---
# ADATA_CFG = (
#     "/ocean/projects/cis240075p/asachan/datasets/TA_muscle/public_datasets/SKM_multimodal_ageing/objects/tmp/myofibers_female.h5ad",
#     "sample", "OM6", "OM9",
#     {"OM6": "#B6D7A8", "OM9": "#F08080"},
#     "human",
#     None,
# )

# --- Mice female single-cell, pre-scored (skip Step 3) — PSC-only path so far ---
# ADATA_CFG = (
#     "/ocean/projects/cis240075p/asachan/datasets/TA_muscle/ERCC1_KO_mice/integrated_samples/objects/female_scored_aucell.h5ad",
#     "condition", "WT", "KO",
#     {"WT": "#B6D7A8", "KO": "#F08080"},
#     "mice",
#     None,
# )

# --- Mice male single-cell, pre-scored (skip Step 3) — PSC-only path so far ---
# ADATA_CFG = (
#     "/ocean/projects/cis240075p/asachan/datasets/TA_muscle/ERCC1_KO_mice/integrated_samples/objects/male_scored_aucell.h5ad",
#     "condition", "WT", "KO",
#     {"WT": "#B6D7A8", "KO": "#F08080"},
#     "mice",
#     None,
# )

adata_path, GROUP_COL, GROUP_1, GROUP_2, PALETTE, GENE_ORIGIN, PREP = ADATA_CFG
GROUPS = (GROUP_1, GROUP_2)

adata = pgl.load_adata(adata_path)
if PREP is not None:
    PREP(adata)
print(adata)
print(f"\n{GROUP_COL} counts:\n{adata.obs[GROUP_COL].value_counts()}")

## Cell B — pick a geneset / pathway

Three flavors: a custom hardcoded list, a single pathway from a GMT, or multiple GMTs stacked.
Uncomment exactly one block. `PATHWAY_NAME` is what becomes the `obs` column after AUCell scoring.

In [ ]:
# --- Option 1: custom hardcoded gene list (atrophy) ---
CUSTOM_ATROPHY_GENES = [
    "UBB", "UBC", "FBXO32", "TRIM63", "MDM2", "FBXO30", "CAMK2B", "TIE1",
    "PSMA1", "PSMA2", "PSMA3", "PSMA4", "PSMA5", "PSMA6", "PSMA7",
    "PSMB1", "PSMB2", "PSMB3", "PSMB4", "FBXO21", "FBXO31", "NEDD4",
    "UBE2B", "UBE2G1", "UBE2J1", "CTSL", "CTSV", "BNIP3", "DEPP1",
    "GABARAPL1", "MAP1LC3", "RETREG1", "SQSTM1", "CAPN1", "CAPN2",
    "ATF4", "FOXO1", "FOXO3A", "HDAC9", "RUNX1", "AMPD3", "CHRNA1",
    "CDKN1A",
]
PATHWAY_NAME = "CUSTOM_ATROPHY"
geneset_df = pgl.geneset_from_list(
    CUSTOM_ATROPHY_GENES,
    name=PATHWAY_NAME,
    gene_origin=GENE_ORIGIN,
)

# --- Option 2: one pathway from a mouse MSigDB GMT ---
# MICE_MSIGDB = "/ocean/projects/cis240075p/asachan/datasets/gene_sets/mouse/msigdb.v2024.1.Mm.symbols.gmt"
# PATHWAY_NAME = "GOBP_PYRUVATE_METABOLIC_PROCESS"
# geneset_df = pgl.geneset_from_gmt(
#     gmt_path=MICE_MSIGDB,
#     include_pathways=[PATHWAY_NAME],
#     gene_origin=GENE_ORIGIN,
# )

# --- Option 3: senescence (SAUL SEN-MAYO UP) ---
# SEN_MAYO_GMT = "/ocean/projects/cis240075p/asachan/datasets/gene_sets/mouse/SAUL_SEN_MAYO_UP_IN_SEN.v2024.1.Mm.gmt"
# PATHWAY_NAME = "SAUL_SEN_MAYO_UP_IN_SEN"
# geneset_df = pgl.geneset_from_gmt(
#     gmt_path=SEN_MAYO_GMT,
#     include_pathways=[PATHWAY_NAME],
#     gene_origin=GENE_ORIGIN,
# )

# --- Option 4: multiple human metabolic pathways (one obs column per pathway) ---
# METABOLIC_GMTS = [
#     "/ocean/projects/cis240075p/asachan/datasets/gene_sets/human/metabolism/GOBP_FATTY_ACID_BETA_OXIDATION.v2024.1.Hs.gmt",
#     "/ocean/projects/cis240075p/asachan/datasets/gene_sets/human/metabolism/KEGG_CITRATE_CYCLE_TCA_CYCLE.v2024.1.Hs.gmt",
#     "/ocean/projects/cis240075p/asachan/datasets/gene_sets/human/metabolism/KEGG_GLYCOLYSIS_GLUCONEOGENESIS.v2024.1.Hs.gmt",
#     "/ocean/projects/cis240075p/asachan/datasets/gene_sets/human/metabolism/KEGG_OXIDATIVE_PHOSPHORYLATION.v2024.1.Hs.gmt",
#     "/ocean/projects/cis240075p/asachan/datasets/gene_sets/human/metabolism/REACTOME_BRANCHED_CHAIN_AMINO_ACID_CATABOLISM.v2024.1.Hs.gmt",
#     "/ocean/projects/cis240075p/asachan/datasets/gene_sets/human/metabolism/REACTOME_GLUTAMATE_AND_GLUTAMINE_METABOLISM.v2024.1.Hs.gmt",
# ]
# PATHWAY_NAME = None  # use first computed pathway for downstream single-score plots
# geneset_df = pgl.geneset_from_gmts(METABOLIC_GMTS, gene_origin=GENE_ORIGIN)

# --- Option 5: DNA damage / repair (mouse) ---
# DNA_DAMAGE_GMTS = [
#     "/ocean/projects/cis240075p/asachan/datasets/gene_sets/mouse/dna_damage/GOBP_DNA_REPAIR.v2024.1.Mm.gmt",
#     "/ocean/projects/cis240075p/asachan/datasets/gene_sets/mouse/dna_damage/GOBP_DNA_DAMAGE_RESPONSE.v2024.1.Mm.gmt",
# ]
# PATHWAY_NAME = None
# geneset_df = pgl.geneset_from_gmts(DNA_DAMAGE_GMTS, gene_origin=GENE_ORIGIN)

print(geneset_df.head())
print(f"\nGeneset entries: {len(geneset_df)}  |  unique pathways: {geneset_df['source'].nunique()}")

## Output directory

In [ ]:
OUTPUT_DIR = Path(
    "/ix/djishnu/Akanksha/analysis_code/snRNA_TA_muscle_analysis/TA_muscle_code/py_scripts/3_geneset_scores/Output"
) / (PATHWAY_NAME or geneset_df['source'].iloc[0])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(OUTPUT_DIR)

## Step 2 — geneset ↔ adata overlap

In [ ]:
overlap = pgl.report_geneset_overlap(
    geneset_df=geneset_df,
    adata=adata,
    pathway_name=PATHWAY_NAME,
)

## Step 3 — AUCell scoring

In [ ]:
pathways = pgl.score_geneset_aucell(
    adata=adata,
    geneset_df=geneset_df,
    raw=False,
    tmin=None,  # set to e.g. 4 if the pathway has very few mapped genes
)
if PATHWAY_NAME is None:
    PATHWAY_NAME = pathways[0]
SCORE_COL = PATHWAY_NAME
print(f"Scored pathways: {pathways}\nUsing SCORE_COL = {SCORE_COL}")

## Step 4 — violin–box of the score across groups

In [ ]:
plot_df, violin_fig = pgl.plot_score_by_group(
    adata=adata,
    score_col=SCORE_COL,
    group_col=GROUP_COL,
    palette=PALETTE,
    group_order=list(GROUPS),
    title=SCORE_COL,
)
violin_fig.savefig(OUTPUT_DIR / f"{SCORE_COL}_violin.png", dpi=300, bbox_inches="tight")
violin_fig.savefig(OUTPUT_DIR / f"{SCORE_COL}_violin.svg", bbox_inches="tight")
violin_fig

## Step 5 — driver genes per group + heatmap

Spearman correlation between every pathway gene and the AUCell score, computed independently per group.

In [ ]:
ranked_dfs = pgl.compute_driver_genes_per_group(
    adata=adata,
    geneset=geneset_df,
    score_col=SCORE_COL,
    group_col=GROUP_COL,
    groups=list(GROUPS),
)
heatmap_df, heatmap_fig, _ = pgl.plot_driver_heatmap(
    ranked_dfs=ranked_dfs,
    top_n=20,
    title=f"Driver Genes \u2014 {SCORE_COL}",
)
heatmap_fig.savefig(OUTPUT_DIR / f"{SCORE_COL}_drivers_heatmap.png", dpi=300, bbox_inches="tight")
heatmap_fig.savefig(OUTPUT_DIR / f"{SCORE_COL}_drivers_heatmap.svg", bbox_inches="tight")
heatmap_fig

## Step 6 — Cliff's delta

When multiple pathways were scored (Option 4 or 5), this produces one row per pathway.

In [ ]:
cliffs_df = pgl.cliffs_delta_table(
    adata=adata,
    score_cols=pathways,
    group_col=GROUP_COL,
    group1=GROUP_1,
    group2=GROUP_2,
    output_csv=OUTPUT_DIR / f"{SCORE_COL}_cliffs_delta.csv",
    output_image=OUTPUT_DIR / f"{SCORE_COL}_cliffs_delta.png",
)
cliffs_df

## (optional) Run everything in one call

For a single pathway, `pgl.run_pipeline` does steps 2–6 in one go and returns every intermediate object.

In [ ]:
# result = pgl.run_pipeline(
#     adata=adata,
#     geneset_df=geneset_df,
#     group_col=GROUP_COL,
#     groups=GROUPS,
#     palette=PALETTE,
#     pathway_name=PATHWAY_NAME,
#     output_dir=OUTPUT_DIR,
#     top_n_drivers=20,
# )
# result.keys()